# Round 1 — Evidence Walkthrough

The cover slide claims *"All figures trace to the project repository."* This notebook is
that claim, made executable: every number in the pitch, recomputed here from committed
files.

**It makes no API calls and needs no keys.** It reads only the evidence already in the
repo, so it costs nothing and cannot fail on stage.

**This is not the live demo.** The demo is the Streamlit dashboard and two browser tabs —
see the last section. Run this before presenting, to confirm the deck's numbers still hold.

Kernel: `bootcamp-env`.  Run: *Kernel → Restart & Run All*.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "dashboard"))
sys.path.insert(0, str(ROOT / "classifier"))
pd.set_option("display.max_colwidth", 60)

def load(p):
    return json.loads((ROOT / p).read_text())

EVIDENCE = [
    "data/complaints_dashboard.csv", "data/complaints_triage.csv.gz",
    "classifier/eval_results_gpt-4o.json", "classifier/eval_results_gpt-4o-mini.json",
    "classifier/ambiguity_results.json", "classifier/taxonomy.json",
    "langsmith/experiment_summary.json", "langsmith/traces_export.json",
    "cost_estimation/cost_model.json", "cost_estimation/token_measurement.json",
]
missing = [p for p in EVIDENCE if not (ROOT / p).exists()]
print("evidence files present:", len(EVIDENCE) - len(missing), "/", len(EVIDENCE))
print("missing:", missing or "none")

evidence files present: 10 / 10
missing: none


## 1 · The corpus

Public CFPB complaint data, curated by `data_prep.py`. Two corrections were applied before
any analysis — the publication-lag window cut, and a dead handling-time metric — both
documented in `research/sector_research.md`.

In [2]:
import metrics as M                     # dashboard/metrics.py — single source of truth

df = M.load()
h  = M.headline(df)

print(f"complaints          {h['complaints']:,}")
print(f"firms               {h['firms']}")
print(f"issue classes       {h['issue_classes']}")
print(f"window              {M.WINDOW_START} to {M.WINDOW_END}")
print(f"median narrative    {h['median_chars']:,.0f} characters (90th pct {h['p90_chars']:,.0f})")

complaints          16,839
firms               567
issue classes       64
window              2026-05-01 to 2026-06-27
median narrative    1,201 characters (90th pct 3,013)


## 2 · Slide 3 — the four numbers

The four figures on the dashboard slide, and the point of the fourth: 46.1% of complaints
sit in five of sixty-four categories, which is what makes a bounded test possible.

These are CFPB figures across 567 firms. They are **not** the client's numbers — Phase 0
measures those.

In [3]:
print(f"complaints analysed        {h['complaints']:,}")
print(f"timely responses           {h['timely_pct']:.1f}%   ({h['untimely_n']} not timely)")
print(f"ended with monetary relief {h['monetary_pct']:.1f}%")
print(f"top-5 category share       {h['top5_share_pct']:.1f}%")

M.top_issues(df, 5)[["Issue", "complaints", "share_pct"]]

complaints analysed        16,839
timely responses           98.0%   (335 not timely)
ended with monetary relief 12.7%
top-5 category share       46.1%


,Issue,complaints,share_pct
0,Managing an account,3344,19.858661
1,Problem with a purchase shown on your statement,1887,11.206129
2,Problem with a lender or other company charging your acc...,949,5.635727
3,Fraud or scam,810,4.810262
4,Closing an account,778,4.620227


## 3 · Appendix — volume does not predict monetary relief

Credit-card complaints end with monetary relief far more often than vehicle-loan
complaints. The deck quotes 5.7×.

In [4]:
p = M.by_product(df)[["short", "complaints", "monetary_pct"]]
ratio = p.loc[p['short'].eq('Credit card'), 'monetary_pct'].iat[0] / \
        p.loc[p['short'].eq('Vehicle loan'), 'monetary_pct'].iat[0]
print(f"credit card vs vehicle loan: {ratio:.1f}x")
p

credit card vs vehicle loan: 5.7x


,short,complaints,monetary_pct
0,Checking / savings,5835,13.967438
1,Credit card,5382,17.168339
2,Money transfer,2551,10.584085
5,Vehicle loan,1523,3.020355
3,Personal loan,1163,3.353396
4,Prepaid card,385,10.909091


## 4 · Slide 4 — the 60 traced decisions

Read back from the committed LangSmith export. Every decision has a recorded outcome and a
reason code; none disappeared into an unexplained bucket.

In [5]:
summary = load("langsmith/experiment_summary.json")
traces  = load("langsmith/traces_export.json")

print("experiment:", summary["experiment"], "| environment:", summary["environment"])
print("runs:", summary["runs"])
print()
for code_, n in summary["reason_code_distribution"].items():
    print(f"  {n:>3}  {code_}")
print()
print("customer_ref join:", summary["customer_ref_join"])
print("records in export:", traces["n"])

experiment: triage-round1-ebb7facb | environment: eval
runs: 60

   52  OK_PROPOSED
    4  REJECT_EVIDENCE_NOT_VERBATIM
    4  REJECT_LOW_CONFIDENCE

customer_ref join: {'joined': 60, 'mismatched': 0, 'missing_one_side': 0, 'join_rate_pct': 100.0}
records in export: 60


## 5 · Slide 4 and appendix — agreement, not accuracy

60.5% is **agreement** with CFPB-derived team labels, not accuracy. Two supporting facts:
the model is self-consistent (~89% on repeated runs), and a model costing ~17× more buys
only about 4 percentage points.

In [6]:
full  = load("classifier/eval_results_gpt-4o.json")
mini  = load("classifier/eval_results_gpt-4o-mini.json")
amb   = load("classifier/ambiguity_results.json")
tok   = load("cost_estimation/token_measurement.json")

a_full = full["HEADLINE_team_accuracy_volume_weighted_pct"]     # key name predates the rename
a_mini = mini["HEADLINE_team_accuracy_volume_weighted_pct"]
print(f"team agreement, gpt-4o        {a_full}%   (n={full['n_evaluated']})")
print(f"team agreement, gpt-4o-mini   {a_mini}%   (n={mini['n_evaluated']})")
print(f"difference                    {a_full - a_mini:.1f} percentage points")
print()
print(f"self-agreement, same team on repeat runs   {amb['self_agreement_team_pct']}%")
print(f"evidence quote genuinely verbatim          {full['evidence_verbatim_pct']}%")
print()
IN, OUT = tok["input_mean"], tok["output_mean"]
cost = lambda pi, po: IN/1e6*pi + OUT/1e6*po
print(f"cost ratio gpt-4o : gpt-4o-mini = {cost(2.50,10.0)/cost(0.15,0.60):.1f}x")

team agreement, gpt-4o        60.5%   (n=240)
team agreement, gpt-4o-mini   56.8%   (n=240)
difference                    3.7 percentage points

self-agreement, same team on repeat runs   89.0%
evidence quote genuinely verbatim          99.6%

cost ratio gpt-4o : gpt-4o-mini = 16.7x


## 6 · Slide 5 — cost, and where it goes

Every figure derives from `cost_estimation/cost_model.py`. The assumptions, each labelled
sourced / measured / assumption / judgement, are in `cost_estimation/assumptions.md`.

In [7]:
cm = load("cost_estimation/cost_model.json")
r  = cm["running_cost"]

print(f"complaints per year (modelled)  {cm['volume']['complaints_per_year']:,}")
print()
print(f"  model calls        EUR {r['api_per_year_eur']:>8.2f}")
print(f"  platform           EUR {r['platform_per_year_eur']:>8,.0f}")
print(f"  consultant review  EUR {r['model_review_per_year_eur']:>8,.0f}")
print(f"  TOTAL              EUR {r['total_per_year_eur']:>8,.0f}")
print()
print(f"  model share of running cost: {100*r['api_per_year_eur']/r['total_per_year_eur']:.4f}%")
print()
print(f"to the pilot decision  EUR {cm['upfront_to_end_of_pilot_eur']:,}")
print(f"full programme         EUR {cm['upfront_full_eur']:,}")
print(f"break-even            {cm['break_even']['avoided_ombudsman_cases_to_cover_running_cost']} avoided ombudsman referrals/year")

complaints per year (modelled)  2,426

  model calls        EUR     0.29
  platform           EUR    1,800
  consultant review  EUR    5,600
  TOTAL              EUR    7,400

  model share of running cost: 0.0039%

to the pilot decision  EUR 14,000
full programme         EUR 24,500
break-even            3.7 avoided ombudsman referrals/year


## 7 · Reconciliation — every deck figure against the evidence

If any row reads **MISMATCH**, the deck and the repository disagree and the deck is wrong
until proven otherwise.

In [8]:
checks = [
    ("Complaints analysed",       f"{h['complaints']:,}",                    "16,839"),
    ("Firms",                     str(h['firms']),                           "567"),
    ("Issue categories",          str(h['issue_classes']),                   "64"),
    ("Timely responses",          f"{h['timely_pct']:.1f}%",                 "98.0%"),
    ("Monetary relief",           f"{h['monetary_pct']:.1f}%",               "12.7%"),
    ("Top-5 concentration",       f"{h['top5_share_pct']:.1f}%",             "46.1%"),
    ("Card vs vehicle relief",    f"{ratio:.1f}x",                           "5.7x"),
    ("Traced decisions",          str(summary['runs']),                      "60"),
    ("Proposed a team",           str(summary['reason_code_distribution']['OK_PROPOSED']), "52"),
    ("Stopped, quote not there",  str(summary['reason_code_distribution']['REJECT_EVIDENCE_NOT_VERBATIM']), "4"),
    ("Stopped, low confidence",   str(summary['reason_code_distribution']['REJECT_LOW_CONFIDENCE']), "4"),
    ("Team agreement (gpt-4o)",   f"{a_full}%",                              "60.5%"),
    ("Self-agreement",            f"{amb['self_agreement_team_pct']}%",      "89.0%"),
    ("Cost ratio between models", f"{cost(2.50,10.0)/cost(0.15,0.60):.0f}x", "17x"),
    ("Model calls per year",      f"EUR {r['api_per_year_eur']:.2f}",        "EUR 0.29"),
    ("Running cost per year",     f"EUR {r['total_per_year_eur']:,}",        "EUR 7,400"),
    ("To the pilot decision",     f"EUR {cm['upfront_to_end_of_pilot_eur']:,}", "EUR 14,000"),
    ("Full programme",            f"EUR {cm['upfront_full_eur']:,}",         "EUR 24,500"),
]
rec = pd.DataFrame(checks, columns=["Figure", "Computed here", "Stated in the deck"])
rec["Status"] = ["OK" if a.replace(",","").lower() == b.replace(",","").lower() else "MISMATCH"
                 for a, b in zip(rec["Computed here"], rec["Stated in the deck"])]
print("MISMATCHES:", (rec.Status == "MISMATCH").sum())
rec

MISMATCHES: 0


,Figure,Computed here,Stated in the deck,Status
0,Complaints analysed,"16,839","16,839",OK
1,Firms,567,567,OK
2,Issue categories,64,64,OK
3,Timely responses,98.0%,98.0%,OK
4,Monetary relief,12.7%,12.7%,OK
5,Top-5 concentration,46.1%,46.1%,OK
6,Card vs vehicle relief,5.7x,5.7x,OK
7,Traced decisions,60,60,OK
8,Proposed a team,52,52,OK
9,"Stopped, quote not there",4,4,OK


## 8 · The live demo — what to open, not what to run here

This notebook proves the numbers. The demo shows the system.

| Slide | What to show | How |
|---|---|---|
| 3 | The dashboard | `streamlit run dashboard/app.py` — start it **before** presenting |
| 4 | The workflow running | n8n workflow `NkRpklvLHKgcP3Ol` on the cohort instance |
| 4 | The same decision, monitored | LangSmith experiment `triage-round1-ebb7facb` (EU workspace) |

Both browser tabs need to be open and logged in beforehand. Every one of these has a
screenshot fallback already on the slide, so a cold demo costs you nothing.

---

### Optional: classify one complaint live

Only if you want to show a classification without n8n. **This one cell costs money and
needs `OPENAI_API_KEY`** — everything above runs free. It is disabled by default; set
`RUN_LIVE = True` to use it.

In [9]:
RUN_LIVE = False        # set True to spend roughly $0.0002 on one classification

if RUN_LIVE:
    import traced_classifier as TC
    row = pd.read_csv(ROOT / "data" / "complaints_triage.csv.gz").sample(1, random_state=11).iloc[0]
    out = TC.classify(row["Complaint ID"], row["Product"], row["Consumer complaint narrative"])
    for k in ("product", "proposed_queue", "proposed_team", "confidence",
              "evidence", "evidence_is_verbatim", "decision", "reason_code"):
        print(f"  {k:<22} {out[k]}")
    print(f"\n  CFPB label for this complaint: {row['Issue']}")
else:
    print("Live classification disabled. Set RUN_LIVE = True to enable.")

Live classification disabled. Set RUN_LIVE = True to enable.
